In [1]:
import pandas as pd 
import numpy as np
import requests

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_American_films_of_2018"

tables = pd.read_html(url, header=0, storage_options={"User-Agent": "Mozilla/5.0"})

df1 = tables[2]
df2 = tables[3]
df3 = tables[4]
df4 = tables[5]

In [3]:
df = pd.concat([df1, df2, df3, df4], ignore_index=True)
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,5,Insidious: The Last Key,Universal Pictures / Blumhouse Productions / S...,Adam Robitel (director); Leigh Whannell (scree...,[2]
1,J A N U A R Y,5,The Strange Ones,Vertical Entertainment,Christopher Radcliff (director/screenplay); La...,[3]
2,J A N U A R Y,12,The Commuter,Lionsgate / StudioCanal / The Picture Company,Jaume Collet-Serra (director); Byron Willinger...,[4]
3,J A N U A R Y,12,Proud Mary,Screen Gems,"Babak Najafi (director); John S. Newman, Chris...",[5]
4,J A N U A R Y,12,Acts of Violence,Lionsgate Premiere,Brett Donowho (director); Nicolas Aaron Mezzan...,[6]
...,...,...,...,...,...,...
246,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[237]
247,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[141]
248,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[116]
249,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[206]


In [4]:
from tmdbv3api import TMDb
import json

tmdb = TMDb()
tmdb.api_key = 'bc6771260f54114bb3e5d552c1776130'

In [5]:
from tmdbv3api import Movie

tmdb_movie = Movie()

def get_genres(x):
    genres = []
    result = tmdb_movie.search(x)
    movie_id = result[0].id
    response = requests.get(f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}')
    data_json = response.json()
    if data_json['genres']:
        genre_str = " "
        for i in range(0, len(data_json['genres'])):
            genres.append(data_json['genres'][i]['name'])
        return genre_str.join(genres)
    else:
        return np.nan


In [6]:
df['genres'] = df['Title'].map(lambda x: get_genres(str(x)))
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres
0,J A N U A R Y,5,Insidious: The Last Key,Universal Pictures / Blumhouse Productions / S...,Adam Robitel (director); Leigh Whannell (scree...,[2],Horror Thriller
1,J A N U A R Y,5,The Strange Ones,Vertical Entertainment,Christopher Radcliff (director/screenplay); La...,[3],Thriller Drama
2,J A N U A R Y,12,The Commuter,Lionsgate / StudioCanal / The Picture Company,Jaume Collet-Serra (director); Byron Willinger...,[4],Action Thriller Mystery
3,J A N U A R Y,12,Proud Mary,Screen Gems,"Babak Najafi (director); John S. Newman, Chris...",[5],Thriller Action Crime
4,J A N U A R Y,12,Acts of Violence,Lionsgate Premiere,Brett Donowho (director); Nicolas Aaron Mezzan...,[6],Action Crime Thriller
...,...,...,...,...,...,...,...
246,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[237],Romance Comedy
247,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[141],Comedy Mystery Crime
248,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[116],Thriller Science Fiction Action Adventure
249,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[206],Drama History


In [14]:
df['Cast and crew'][1]

'Christopher Radcliff (director/screenplay); Lauren Wolkstein (director); Alex Pettyfer, James Freedson-Jackson, Emily Althaus, Gene Jones, Owen Campbell, Tobias Campbell'

In [18]:
def get_director(x):
    directors = []
    parts = x.split("; ")
    
    for p in parts:
        if "(director)" in p or "(directors)" in p or "(director/screenplay)" in p:
            directors.append(p.split(" (")[0])
    
    if len(directors) == 0:
        return np.nan
    
    return ", ".join(directors)

In [20]:
df['director_name'] = df['Cast and crew'].map(lambda x: get_director(x))

In [21]:
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres,director_name
0,J A N U A R Y,5,Insidious: The Last Key,Universal Pictures / Blumhouse Productions / S...,Adam Robitel (director); Leigh Whannell (scree...,[2],Horror Thriller,Adam Robitel
1,J A N U A R Y,5,The Strange Ones,Vertical Entertainment,Christopher Radcliff (director/screenplay); La...,[3],Thriller Drama,"Christopher Radcliff, Lauren Wolkstein"
2,J A N U A R Y,12,The Commuter,Lionsgate / StudioCanal / The Picture Company,Jaume Collet-Serra (director); Byron Willinger...,[4],Action Thriller Mystery,Jaume Collet-Serra
3,J A N U A R Y,12,Proud Mary,Screen Gems,"Babak Najafi (director); John S. Newman, Chris...",[5],Thriller Action Crime,Babak Najafi
4,J A N U A R Y,12,Acts of Violence,Lionsgate Premiere,Brett Donowho (director); Nicolas Aaron Mezzan...,[6],Action Crime Thriller,Brett Donowho
...,...,...,...,...,...,...,...,...
246,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[237],Romance Comedy,Peter Segal
247,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[141],Comedy Mystery Crime,Etan Cohen
248,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[116],Thriller Science Fiction Action Adventure,Adam McKay
249,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[206],Drama History,Mimi Leder


In [22]:
def get_actor1(x):
    parts = x.split("; ")
    
    for p in parts:
        if "(" not in p:
            actors = p.split(", ")
            if len(actors) > 0:
                return actors[0]
    
    return np.nan

In [23]:
df['actor_1_name'] = df['Cast and crew'].map(lambda x: get_actor1(x))
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres,director_name,actor_1_name
0,J A N U A R Y,5,Insidious: The Last Key,Universal Pictures / Blumhouse Productions / S...,Adam Robitel (director); Leigh Whannell (scree...,[2],Horror Thriller,Adam Robitel,Lin Shaye
1,J A N U A R Y,5,The Strange Ones,Vertical Entertainment,Christopher Radcliff (director/screenplay); La...,[3],Thriller Drama,"Christopher Radcliff, Lauren Wolkstein",Alex Pettyfer
2,J A N U A R Y,12,The Commuter,Lionsgate / StudioCanal / The Picture Company,Jaume Collet-Serra (director); Byron Willinger...,[4],Action Thriller Mystery,Jaume Collet-Serra,Liam Neeson
3,J A N U A R Y,12,Proud Mary,Screen Gems,"Babak Najafi (director); John S. Newman, Chris...",[5],Thriller Action Crime,Babak Najafi,Taraji P. Henson
4,J A N U A R Y,12,Acts of Violence,Lionsgate Premiere,Brett Donowho (director); Nicolas Aaron Mezzan...,[6],Action Crime Thriller,Brett Donowho,Bruce Willis
...,...,...,...,...,...,...,...,...,...
246,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[237],Romance Comedy,Peter Segal,Jennifer Lopez
247,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[141],Comedy Mystery Crime,Etan Cohen,Will Ferrell
248,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[116],Thriller Science Fiction Action Adventure,Adam McKay,Christian Bale
249,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[206],Drama History,Mimi Leder,Felicity Jones


In [25]:
df['actor_1_name'][1]

'Alex Pettyfer'

In [26]:
def get_actor2(x):
    parts = x.split("; ")
    
    for p in parts:
        if "(" not in p:
            actors = p.split(", ")
            if len(actors) > 1:
                return actors[1]
    
    return np.nan

In [27]:
df['actor_2_name'] = df['Cast and crew'].map(lambda x: get_actor2(x))

In [28]:
def get_actor3(x):
    parts = x.split("; ")
    
    for p in parts:
        if "(" not in p:
            actors = p.split(", ")
            if len(actors) > 2:
                return actors[2]
    
    return np.nan

In [29]:
df['actor_3_name'] = df['Cast and crew'].map(lambda x: get_actor3(x))

In [30]:
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres,director_name,actor_1_name,actor_2_name,actor_3_name
0,J A N U A R Y,5,Insidious: The Last Key,Universal Pictures / Blumhouse Productions / S...,Adam Robitel (director); Leigh Whannell (scree...,[2],Horror Thriller,Adam Robitel,Lin Shaye,Angus Sampson,Leigh Whannell
1,J A N U A R Y,5,The Strange Ones,Vertical Entertainment,Christopher Radcliff (director/screenplay); La...,[3],Thriller Drama,"Christopher Radcliff, Lauren Wolkstein",Alex Pettyfer,James Freedson-Jackson,Emily Althaus
2,J A N U A R Y,12,The Commuter,Lionsgate / StudioCanal / The Picture Company,Jaume Collet-Serra (director); Byron Willinger...,[4],Action Thriller Mystery,Jaume Collet-Serra,Liam Neeson,Vera Farmiga,Patrick Wilson
3,J A N U A R Y,12,Proud Mary,Screen Gems,"Babak Najafi (director); John S. Newman, Chris...",[5],Thriller Action Crime,Babak Najafi,Taraji P. Henson,Jahi Di'Allo Winston,Billy Brown
4,J A N U A R Y,12,Acts of Violence,Lionsgate Premiere,Brett Donowho (director); Nicolas Aaron Mezzan...,[6],Action Crime Thriller,Brett Donowho,Bruce Willis,Cole Hauser,Shawn Ashmore
...,...,...,...,...,...,...,...,...,...,...,...
246,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[237],Romance Comedy,Peter Segal,Jennifer Lopez,Leah Remini,Vanessa Hudgens
247,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[141],Comedy Mystery Crime,Etan Cohen,Will Ferrell,John C. Reilly,Rebecca Hall
248,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[116],Thriller Science Fiction Action Adventure,Adam McKay,Christian Bale,Amy Adams,Steve Carell
249,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[206],Drama History,Mimi Leder,Felicity Jones,Armie Hammer,Justin Theroux


In [31]:
df = df.loc[:, ["director_name", "actor_1_name", "actor_2_name", "actor_3_name", "genres", "Title"]]
df.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,Title
0,Adam Robitel,Lin Shaye,Angus Sampson,Leigh Whannell,Horror Thriller,Insidious: The Last Key
1,"Christopher Radcliff, Lauren Wolkstein",Alex Pettyfer,James Freedson-Jackson,Emily Althaus,Thriller Drama,The Strange Ones
2,Jaume Collet-Serra,Liam Neeson,Vera Farmiga,Patrick Wilson,Action Thriller Mystery,The Commuter
3,Babak Najafi,Taraji P. Henson,Jahi Di'Allo Winston,Billy Brown,Thriller Action Crime,Proud Mary
4,Brett Donowho,Bruce Willis,Cole Hauser,Shawn Ashmore,Action Crime Thriller,Acts of Violence


In [32]:
df = df.rename(columns={"Title": "movie_title"})
df.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title
0,Adam Robitel,Lin Shaye,Angus Sampson,Leigh Whannell,Horror Thriller,Insidious: The Last Key
1,"Christopher Radcliff, Lauren Wolkstein",Alex Pettyfer,James Freedson-Jackson,Emily Althaus,Thriller Drama,The Strange Ones
2,Jaume Collet-Serra,Liam Neeson,Vera Farmiga,Patrick Wilson,Action Thriller Mystery,The Commuter
3,Babak Najafi,Taraji P. Henson,Jahi Di'Allo Winston,Billy Brown,Thriller Action Crime,Proud Mary
4,Brett Donowho,Bruce Willis,Cole Hauser,Shawn Ashmore,Action Crime Thriller,Acts of Violence


In [33]:
df.isnull().sum()

director_name     8
actor_1_name      3
actor_2_name      6
actor_3_name     18
genres            3
movie_title       0
dtype: int64

In [34]:
df['director_name'] = df['director_name'].fillna("unknown")
df['actor_1_name'] = df['actor_1_name'].fillna("unknown")
df['actor_2_name'] = df['actor_2_name'].fillna("unknown")
df['actor_3_name'] = df['actor_3_name'].fillna("unknown")
df['genres'] = df['genres'].fillna("unknown")

In [36]:
df2018 = df.copy()
df2018

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title
0,Adam Robitel,Lin Shaye,Angus Sampson,Leigh Whannell,Horror Thriller,Insidious: The Last Key
1,"Christopher Radcliff, Lauren Wolkstein",Alex Pettyfer,James Freedson-Jackson,Emily Althaus,Thriller Drama,The Strange Ones
2,Jaume Collet-Serra,Liam Neeson,Vera Farmiga,Patrick Wilson,Action Thriller Mystery,The Commuter
3,Babak Najafi,Taraji P. Henson,Jahi Di'Allo Winston,Billy Brown,Thriller Action Crime,Proud Mary
4,Brett Donowho,Bruce Willis,Cole Hauser,Shawn Ashmore,Action Crime Thriller,Acts of Violence
...,...,...,...,...,...,...
246,Peter Segal,Jennifer Lopez,Leah Remini,Vanessa Hudgens,Romance Comedy,Second Act
247,Etan Cohen,Will Ferrell,John C. Reilly,Rebecca Hall,Comedy Mystery Crime,Holmes & Watson
248,Adam McKay,Christian Bale,Amy Adams,Steve Carell,Thriller Science Fiction Action Adventure,Vice
249,Mimi Leder,Felicity Jones,Armie Hammer,Justin Theroux,Drama History,On the Basis of Sex


In [37]:
url = "https://en.wikipedia.org/wiki/List_of_American_films_of_2019"

tables = pd.read_html(url, header=0, storage_options={"User-Agent": "Mozilla/5.0"})

df1 = tables[2]
df2 = tables[3]
df3 = tables[4]
df4 = tables[5]

In [38]:
df = pd.concat([df1, df2, df3, df4], ignore_index=True)
df 

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,4,Escape Room,Columbia Pictures / Original Film,"Adam Robitel (director); Bragi F. Schut, Maria...",[2]
1,J A N U A R Y,4,Rust Creek,IFC Films / Lunacy Productions,Jen McGowan (director); Julie Lipson (screenpl...,[3]
2,J A N U A R Y,4,American Hangman,Hangman Justice Productions,Wilson Coneybeare (director/screenplay); Donal...,[4]
3,J A N U A R Y,11,A Dog's Way Home,Columbia Pictures,Charles Martin Smith (director); W. Bruce Came...,[5]
4,J A N U A R Y,11,The Upside,STX Entertainment,Neil Burger (director); Jon Hartmere (screenpl...,[6]
...,...,...,...,...,...,...
246,D E C E M B E R,25,Spies in Disguise,20th Century Fox Animation / Blue Sky Studios ...,"Nick Bruno, Troy Quane (directors); Brad Copel...",[133]
247,D E C E M B E R,25,Little Women,Columbia Pictures / Regency Enterprises,Greta Gerwig (director/screenplay); Saoirse Ro...,[227]
248,D E C E M B E R,25,1917,Universal Pictures / DreamWorks Pictures,Sam Mendes (director/screenplay); Krysty Wilso...,[228]
249,D E C E M B E R,25,Just Mercy,Warner Bros. Pictures / Participant,"Destin Daniel Cretton (director/screenplay), A...",[229]


In [39]:
df['genres'] = df['Title'].map(lambda x: get_genres(str(x)))

In [40]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [41]:
df['director_name'] = df['Cast and crew'].map(lambda x: get_director(str(x)))
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres,director_name
0,J A N U A R Y,4,Escape Room,Columbia Pictures / Original Film,"Adam Robitel (director); Bragi F. Schut, Maria...",[2],Horror Thriller Mystery,Adam Robitel
1,J A N U A R Y,4,Rust Creek,IFC Films / Lunacy Productions,Jen McGowan (director); Julie Lipson (screenpl...,[3],Thriller Drama Action Crime,Jen McGowan
2,J A N U A R Y,4,American Hangman,Hangman Justice Productions,Wilson Coneybeare (director/screenplay); Donal...,[4],Thriller,Wilson Coneybeare
3,J A N U A R Y,11,A Dog's Way Home,Columbia Pictures,Charles Martin Smith (director); W. Bruce Came...,[5],Drama Adventure Family,Charles Martin Smith
4,J A N U A R Y,11,The Upside,STX Entertainment,Neil Burger (director); Jon Hartmere (screenpl...,[6],Comedy Drama,Neil Burger
...,...,...,...,...,...,...,...,...
246,D E C E M B E R,25,Spies in Disguise,20th Century Fox Animation / Blue Sky Studios ...,"Nick Bruno, Troy Quane (directors); Brad Copel...",[133],Animation Action Adventure Comedy Family,"Nick Bruno, Troy Quane"
247,D E C E M B E R,25,Little Women,Columbia Pictures / Regency Enterprises,Greta Gerwig (director/screenplay); Saoirse Ro...,[227],Drama Romance History,Greta Gerwig
248,D E C E M B E R,25,1917,Universal Pictures / DreamWorks Pictures,Sam Mendes (director/screenplay); Krysty Wilso...,[228],War History,Sam Mendes
249,D E C E M B E R,25,Just Mercy,Warner Bros. Pictures / Participant,"Destin Daniel Cretton (director/screenplay), A...",[229],Drama Crime History,Destin Daniel Cretton


In [42]:
def get_actor1(x):
    return ((x.split("screenplay); ")[-1]).split(", ")[0])

In [43]:
df['actor_1_name'] = df['Cast and crew'].map(lambda x: get_actor1(str(x)))
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres,director_name,actor_1_name
0,J A N U A R Y,4,Escape Room,Columbia Pictures / Original Film,"Adam Robitel (director); Bragi F. Schut, Maria...",[2],Horror Thriller Mystery,Adam Robitel,Taylor Russell
1,J A N U A R Y,4,Rust Creek,IFC Films / Lunacy Productions,Jen McGowan (director); Julie Lipson (screenpl...,[3],Thriller Drama Action Crime,Jen McGowan,Hermione Corfield
2,J A N U A R Y,4,American Hangman,Hangman Justice Productions,Wilson Coneybeare (director/screenplay); Donal...,[4],Thriller,Wilson Coneybeare,Donald Sutherland
3,J A N U A R Y,11,A Dog's Way Home,Columbia Pictures,Charles Martin Smith (director); W. Bruce Came...,[5],Drama Adventure Family,Charles Martin Smith,Bryce Dallas Howard
4,J A N U A R Y,11,The Upside,STX Entertainment,Neil Burger (director); Jon Hartmere (screenpl...,[6],Comedy Drama,Neil Burger,Bryan Cranston
...,...,...,...,...,...,...,...,...,...
246,D E C E M B E R,25,Spies in Disguise,20th Century Fox Animation / Blue Sky Studios ...,"Nick Bruno, Troy Quane (directors); Brad Copel...",[133],Animation Action Adventure Comedy Family,"Nick Bruno, Troy Quane",Will Smith
247,D E C E M B E R,25,Little Women,Columbia Pictures / Regency Enterprises,Greta Gerwig (director/screenplay); Saoirse Ro...,[227],Drama Romance History,Greta Gerwig,Saoirse Ronan
248,D E C E M B E R,25,1917,Universal Pictures / DreamWorks Pictures,Sam Mendes (director/screenplay); Krysty Wilso...,[228],War History,Sam Mendes,George MacKay
249,D E C E M B E R,25,Just Mercy,Warner Bros. Pictures / Participant,"Destin Daniel Cretton (director/screenplay), A...",[229],Drama Crime History,Destin Daniel Cretton,Michael B. Jordan


In [50]:
def get_actor2(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 2:
        return np.nan
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[1])

In [51]:
df['actor_2_name'] = df['Cast and crew'].map(lambda x: get_actor2(x))
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres,director_name,actor_1_name,actor_2_name
0,J A N U A R Y,4,Escape Room,Columbia Pictures / Original Film,"Adam Robitel (director); Bragi F. Schut, Maria...",[2],Horror Thriller Mystery,Adam Robitel,Taylor Russell,Logan Miller
1,J A N U A R Y,4,Rust Creek,IFC Films / Lunacy Productions,Jen McGowan (director); Julie Lipson (screenpl...,[3],Thriller Drama Action Crime,Jen McGowan,Hermione Corfield,Jay Paulson
2,J A N U A R Y,4,American Hangman,Hangman Justice Productions,Wilson Coneybeare (director/screenplay); Donal...,[4],Thriller,Wilson Coneybeare,Donald Sutherland,Vincent Kartheiser
3,J A N U A R Y,11,A Dog's Way Home,Columbia Pictures,Charles Martin Smith (director); W. Bruce Came...,[5],Drama Adventure Family,Charles Martin Smith,Bryce Dallas Howard,Edward James Olmos
4,J A N U A R Y,11,The Upside,STX Entertainment,Neil Burger (director); Jon Hartmere (screenpl...,[6],Comedy Drama,Neil Burger,Bryan Cranston,Kevin Hart
...,...,...,...,...,...,...,...,...,...,...
246,D E C E M B E R,25,Spies in Disguise,20th Century Fox Animation / Blue Sky Studios ...,"Nick Bruno, Troy Quane (directors); Brad Copel...",[133],Animation Action Adventure Comedy Family,"Nick Bruno, Troy Quane",Will Smith,Tom Holland
247,D E C E M B E R,25,Little Women,Columbia Pictures / Regency Enterprises,Greta Gerwig (director/screenplay); Saoirse Ro...,[227],Drama Romance History,Greta Gerwig,Saoirse Ronan,Emma Watson
248,D E C E M B E R,25,1917,Universal Pictures / DreamWorks Pictures,Sam Mendes (director/screenplay); Krysty Wilso...,[228],War History,Sam Mendes,George MacKay,Dean-Charles Chapman
249,D E C E M B E R,25,Just Mercy,Warner Bros. Pictures / Participant,"Destin Daniel Cretton (director/screenplay), A...",[229],Drama Crime History,Destin Daniel Cretton,Michael B. Jordan,Jamie Foxx


In [52]:
def get_actor3(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 3:
        return np.nan
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[2])

In [53]:
df['actor_3_name'] = df['Cast and crew'].map(lambda x: get_actor3(x))
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres,director_name,actor_1_name,actor_2_name,actor_3_name
0,J A N U A R Y,4,Escape Room,Columbia Pictures / Original Film,"Adam Robitel (director); Bragi F. Schut, Maria...",[2],Horror Thriller Mystery,Adam Robitel,Taylor Russell,Logan Miller,Deborah Ann Woll
1,J A N U A R Y,4,Rust Creek,IFC Films / Lunacy Productions,Jen McGowan (director); Julie Lipson (screenpl...,[3],Thriller Drama Action Crime,Jen McGowan,Hermione Corfield,Jay Paulson,Sean O'Bryan
2,J A N U A R Y,4,American Hangman,Hangman Justice Productions,Wilson Coneybeare (director/screenplay); Donal...,[4],Thriller,Wilson Coneybeare,Donald Sutherland,Vincent Kartheiser,Oliver Dennis
3,J A N U A R Y,11,A Dog's Way Home,Columbia Pictures,Charles Martin Smith (director); W. Bruce Came...,[5],Drama Adventure Family,Charles Martin Smith,Bryce Dallas Howard,Edward James Olmos,Alexandra Shipp
4,J A N U A R Y,11,The Upside,STX Entertainment,Neil Burger (director); Jon Hartmere (screenpl...,[6],Comedy Drama,Neil Burger,Bryan Cranston,Kevin Hart,Nicole Kidman
...,...,...,...,...,...,...,...,...,...,...,...
246,D E C E M B E R,25,Spies in Disguise,20th Century Fox Animation / Blue Sky Studios ...,"Nick Bruno, Troy Quane (directors); Brad Copel...",[133],Animation Action Adventure Comedy Family,"Nick Bruno, Troy Quane",Will Smith,Tom Holland,Rashida Jones
247,D E C E M B E R,25,Little Women,Columbia Pictures / Regency Enterprises,Greta Gerwig (director/screenplay); Saoirse Ro...,[227],Drama Romance History,Greta Gerwig,Saoirse Ronan,Emma Watson,Florence Pugh
248,D E C E M B E R,25,1917,Universal Pictures / DreamWorks Pictures,Sam Mendes (director/screenplay); Krysty Wilso...,[228],War History,Sam Mendes,George MacKay,Dean-Charles Chapman,Mark Strong
249,D E C E M B E R,25,Just Mercy,Warner Bros. Pictures / Participant,"Destin Daniel Cretton (director/screenplay), A...",[229],Drama Crime History,Destin Daniel Cretton,Michael B. Jordan,Jamie Foxx,Brie Larson


In [54]:
# Selecting only the relevant columns for our recommendation system

df = df.loc[:, ["director_name", "actor_1_name", "actor_2_name", "actor_3_name", "genres", "Title"]]
df.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,Title
0,Adam Robitel,Taylor Russell,Logan Miller,Deborah Ann Woll,Horror Thriller Mystery,Escape Room
1,Jen McGowan,Hermione Corfield,Jay Paulson,Sean O'Bryan,Thriller Drama Action Crime,Rust Creek
2,Wilson Coneybeare,Donald Sutherland,Vincent Kartheiser,Oliver Dennis,Thriller,American Hangman
3,Charles Martin Smith,Bryce Dallas Howard,Edward James Olmos,Alexandra Shipp,Drama Adventure Family,A Dog's Way Home
4,Neil Burger,Bryan Cranston,Kevin Hart,Nicole Kidman,Comedy Drama,The Upside


In [55]:
df = df.rename(columns={"Title": "movie_title"})
df.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title
0,Adam Robitel,Taylor Russell,Logan Miller,Deborah Ann Woll,Horror Thriller Mystery,Escape Room
1,Jen McGowan,Hermione Corfield,Jay Paulson,Sean O'Bryan,Thriller Drama Action Crime,Rust Creek
2,Wilson Coneybeare,Donald Sutherland,Vincent Kartheiser,Oliver Dennis,Thriller,American Hangman
3,Charles Martin Smith,Bryce Dallas Howard,Edward James Olmos,Alexandra Shipp,Drama Adventure Family,A Dog's Way Home
4,Neil Burger,Bryan Cranston,Kevin Hart,Nicole Kidman,Comedy Drama,The Upside


In [56]:
df.isnull().sum()

director_name     0
actor_1_name      0
actor_2_name      1
actor_3_name     17
genres            0
movie_title       0
dtype: int64

In [58]:
df['actor_2_name'] = df['actor_2_name'].fillna("unknown")
df['actor_3_name'] = df['actor_3_name'].fillna("unknown")

In [59]:
df2019 = df.copy()
df2019

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title
0,Adam Robitel,Taylor Russell,Logan Miller,Deborah Ann Woll,Horror Thriller Mystery,Escape Room
1,Jen McGowan,Hermione Corfield,Jay Paulson,Sean O'Bryan,Thriller Drama Action Crime,Rust Creek
2,Wilson Coneybeare,Donald Sutherland,Vincent Kartheiser,Oliver Dennis,Thriller,American Hangman
3,Charles Martin Smith,Bryce Dallas Howard,Edward James Olmos,Alexandra Shipp,Drama Adventure Family,A Dog's Way Home
4,Neil Burger,Bryan Cranston,Kevin Hart,Nicole Kidman,Comedy Drama,The Upside
...,...,...,...,...,...,...
246,"Nick Bruno, Troy Quane",Will Smith,Tom Holland,Rashida Jones,Animation Action Adventure Comedy Family,Spies in Disguise
247,Greta Gerwig,Saoirse Ronan,Emma Watson,Florence Pugh,Drama Romance History,Little Women
248,Sam Mendes,George MacKay,Dean-Charles Chapman,Mark Strong,War History,1917
249,Destin Daniel Cretton,Michael B. Jordan,Jamie Foxx,Brie Larson,Drama Crime History,Just Mercy


In [60]:
df2018['movie_title'] = df2018['movie_title'].str.lower()
df2019['movie_title'] = df2019['movie_title'].str.lower()

In [61]:
df2018['comb'] = df2018['actor_1_name'] + ' ' + df2018['actor_2_name'] + ' ' + df2018['actor_3_name'] + ' ' + df2018['director_name'] + ' ' + df2018['genres']
df2019['comb'] = df2019['actor_1_name'] + ' ' + df2019['actor_2_name'] + ' ' + df2019['actor_3_name'] + ' ' + df2019['director_name'] + ' ' + df2019['genres']

In [62]:
old = pd.read_csv("../Data/processed/processed_combined_movie_metadata.csv")
old.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...


In [64]:
final_df = pd.concat([old, df2018, df2019], ignore_index=True)
final_df.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...


In [65]:
final_df.isnull().sum()

director_name    0
actor_1_name     0
actor_2_name     0
actor_3_name     0
genres           0
movie_title      0
comb             0
dtype: int64

In [66]:
final_df.duplicated().sum()

np.int64(0)

In [67]:
final_df.to_csv("../Data/processed/final_combined_movie_metadata.csv", index=False)